# Is the answer correct? Judged by Liquid AI's d1

Stage 3 of the eval suite: the whole pipeline, judged on whether its answers are right. The earlier stages never check
that. Faithfulness, at its default setting, only catches claims that contradict the retrieved chunks, and Answer
Relevancy only checks that the answer stays on the question. Here each answer is compared with the golden set's
reference answer, written from the PDF.

Nothing is answered again. The answers are the 15 that the agent (`deepseek-flash`) gave in stage 2, saved in
`generator_answers.json`, and the reference answers are those in `evals/goldens/retriever_goldens.json`. So no DeepSeek
call is made. Liquid AI's `d1.1` judges each pair through deepeval's `JevEval` metric, with deepeval's `TypeSafeModel`
pointed at Liquid's API (`https://api.liquid.ai/decisions`) and the key `LIQUID_API_KEY` from `.env`, one request per answer.

The "Correctness" metric asks d1 three questions about each answer:

| Question to d1 | Type | Weight |
|---|---|---|
| The answer contains no fact that contradicts the reference answer. | yes or no | 2 |
| The answer states every fact in the reference answer. | yes or no | 1 |
| How closely does the answer match the reference answer? | score: contradicts it, mostly wrong, mostly right, fully matches | 1 |

Each question's value runs from 0 to 1: d1's probability of "yes" for a yes-or-no question, and for the score the level
d1 expects, from 0 for "contradicts it" to 1 for "fully matches". The metric's score is their weighted mean, so a
contradiction counts twice as much as a missing fact. An answer may add facts the reference does not have; only
contradicting the reference's facts, or leaving them out, counts against it.

In [1]:
import json
import os
import sys
import time
from pathlib import Path

# Set before deepeval is imported, since it reads it then
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"  # deepeval sends no usage data

PROJECT_ROOT = Path.cwd().parents[1]  # this notebook lives in evals/notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

from deepeval.metrics import JevEval
from deepeval.metrics.jev_eval import Noul, Score
from deepeval.models import TypeSafeModel
from deepeval.test_case import LLMTestCase, SingleTurnParams
from IPython.display import Markdown, display

from multimodal_rag import config  # loads .env, with LIQUID_API_KEY

GOLDENS_PATH = PROJECT_ROOT / "evals" / "goldens" / "retriever_goldens.json"  # the questions and reference answers
ANSWERS_PATH = PROJECT_ROOT / "evals" / "notebooks" / "generator_answers.json"  # stage 2's answers

# Judge: Liquid AI's decision model, reached through the TypeSafe SDK. Its key is LIQUID_API_KEY in .env.
JUDGE_MODEL = "d1.1"
LIQUID_BASE_URL = "https://api.liquid.ai/decisions"

goldens = json.loads(GOLDENS_PATH.read_text(encoding="utf-8"))
answers = json.loads(ANSWERS_PATH.read_text(encoding="utf-8"))
assert [answer["question"] for answer in answers] == [golden["input"] for golden in goldens], "the saved answers must be for these questions"
print(f"{len(answers)} answers by {sorted({answer['model'] for answer in answers})}, judged by {JUDGE_MODEL}")

15 answers by ['deepseek-flash'], judged by d1.1


## 1. Judge

One metric serves every answer, since its questions refer only to `actual_output` and `expected_output`. d1 also sees
the question (`input`), and answers all three questions in one request.

In [2]:
d1 = TypeSafeModel(model=JUDGE_MODEL, api_key=os.environ["LIQUID_API_KEY"], base_url=LIQUID_BASE_URL)
correctness = JevEval(
    name="Correctness",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
    questions=[
        Noul("actual_output contains no fact that contradicts expected_output.", weight=2),
        Noul("actual_output states every fact in expected_output."),
        Score(
            "How closely does actual_output match expected_output?",
            levels=["Contradicts it", "Mostly wrong", "Mostly right", "Fully matches"],
        ),
    ],
    system_one_model=d1,
    async_mode=False,
)

results = []  # for each answer, the metric's score and its outcome for each question
start = time.perf_counter()
for golden, answer in zip(goldens, answers):
    test_case = LLMTestCase(input=golden["input"], actual_output=answer["answer"], expected_output=golden["expected_output"])
    correctness.measure(test_case, _show_indicator=False)
    results.append({"score": correctness.score, "outcomes": correctness.score_breakdown, "reason": correctness.reason})
print(f"d1 judged {len(results)} answers in {time.perf_counter() - start:.0f} seconds")

d1 judged 15 answers in 9 seconds


## 2. Results

"No contradiction" and "Every fact" are d1's probabilities that the statement is true. "Closeness" is the level d1
thought most likely, with its probability. "Correctness" is the metric's score.

In [3]:
def most_likely(outcome):
    level, probability = max(outcome["probabilities"].items(), key=lambda item: item[1])
    return f"{level} ({probability:.2f})"


rows = [
    "| # | Content | Question | No contradiction | Every fact | Closeness | Correctness |",
    "|---|---|---|---|---|---|---|",
]
by_content = {}  # content type -> list of scores
for number, (golden, result) in enumerate(zip(goldens, results), start=1):
    no_contradiction, every_fact, closeness = result["outcomes"]
    content = golden["additional_metadata"]["content"]
    rows.append(
        f"| {number} | {content} | {golden['input']} | {no_contradiction['value']:.2f} | {every_fact['value']:.2f} "
        f"| {most_likely(closeness)} | {result['score']:.2f} |"
    )
    by_content.setdefault(content, []).append(result["score"])
display(Markdown("\n".join(rows)))

summary = ["| Content | Questions | Mean correctness |", "|---|---|---|"]
for content, content_scores in by_content.items():
    summary.append(f"| {content} | {len(content_scores)} | {sum(content_scores) / len(content_scores):.2f} |")
all_scores = [result["score"] for result in results]
summary.append(f"| **all** | {len(all_scores)} | **{sum(all_scores) / len(all_scores):.2f}** |")
display(Markdown("\n".join(summary)))

| # | Content | Question | No contradiction | Every fact | Closeness | Correctness |
|---|---|---|---|---|---|---|
| 1 | text | What did NVIDIA get from its December 2025 agreement with Groq, and how much did it pay? | 0.96 | 0.96 | Fully matches (0.88) | 0.96 |
| 2 | text | How much room does NVIDIA still have to buy back its own shares? | 0.99 | 0.97 | Fully matches (0.87) | 0.98 |
| 3 | text | Why did NVIDIA's gross margin go down in fiscal 2026? | 0.99 | 0.98 | Fully matches (0.93) | 0.98 |
| 4 | text | How big is NVIDIA's workforce, and how many people work in research? | 0.96 | 0.86 | Fully matches (0.79) | 0.93 |
| 5 | text | When and where is the 2026 annual shareholder meeting, and who can vote at it? | 0.98 | 0.93 | Fully matches (0.84) | 0.96 |
| 6 | table | What did NVIDIA pay its auditor PwC in fiscal 2026, and how much of that was for the audit itself? | 0.99 | 0.03 | Mostly right (0.73) | 0.68 |
| 7 | table | How much of NVIDIA's Data Center revenue came from compute and how much from networking? | 0.93 | 0.07 | Mostly right (0.70) | 0.65 |
| 8 | table | How much revenue came from customers based in China, and how did that change from the year before? | 0.82 | 0.12 | Mostly right (0.76) | 0.61 |
| 9 | table | How much were NVIDIA's non-employee directors paid for fiscal 2026? | 0.92 | 0.94 | Fully matches (0.77) | 0.92 |
| 10 | table | Who are NVIDIA's largest shareholders? | 0.95 | 0.93 | Fully matches (0.82) | 0.94 |
| 11 | chart | Which organizations publish the most open-source AI repositories, according to the report? | 0.85 | 0.83 | Fully matches (0.52) | 0.84 |
| 12 | chart | How much cash has NVIDIA returned to shareholders over the past one, three and five years? | 0.97 | 0.94 | Fully matches (0.82) | 0.96 |
| 13 | chart | How is the CEO's target pay for fiscal 2026 split between salary, cash and stock awards? | 0.65 | 0.11 | Mostly right (0.70) | 0.52 |
| 14 | multi_page | Who chairs each of the Board's committees, and how often did each committee meet in fiscal 2026? | 0.81 | 0.04 | Mostly right (0.82) | 0.58 |
| 15 | multi_page | How much did NVIDIA hold in cash equivalents and marketable securities at the end of fiscal 2026, compared with a year earlier? | 0.47 | 0.24 | Mostly right (0.57) | 0.44 |

| Content | Questions | Mean correctness |
|---|---|---|
| text | 5 | 0.96 |
| table | 5 | 0.76 |
| chart | 3 | 0.77 |
| multi_page | 2 | 0.51 |
| **all** | 15 | **0.80** |

## 3. Answers and references

Each answer next to its reference answer, with d1's reason: its answer to each question, the probability and
confidence behind it, and how the score follows.

In [4]:
for number, (golden, answer, result) in enumerate(zip(goldens, answers, results), start=1):
    display(Markdown(
        f"**{number}. {golden['input']}**\n\n"
        f"*Reference:* {golden['expected_output']}\n\n"
        f"*Answer:*\n\n{answer['answer']}\n\n"
        f"*d1:* {result['reason'].replace(chr(10), '<br>')}\n\n---"
    ))

**1. What did NVIDIA get from its December 2025 agreement with Groq, and how much did it pay?**

*Reference:* In December 2025, NVIDIA entered into a non-exclusive license agreement for Groq's language processing unit technology and hired certain Groq employees; it bought no customer contracts, existing products or equity interests. Total consideration was $13.0 billion paid at closing plus $4 billion, inclusive of imputed interest, payable within one year. NVIDIA recorded $14.4 billion of goodwill and a $2.5 billion developed technology intangible asset with a five-year useful life.

*Answer:*

**What NVIDIA obtained (page 149):** Under a **non-exclusive license agreement with Groq, Inc. entered into in December 2025**, NVIDIA licensed Groq's **language processing unit (LPU) technology** and **hired certain Groq employees**. Notably, **no customer contracts, existing products, or equity interests were purchased**.

**What it paid / total consideration (page 149):** Total consideration was **$17.0 billion**, consisting of:
- **$13.0 billion paid at closing**, and
- **$4 billion** (inclusive of imputed interest) **payable within one year**, recorded in Accrued and Other Current Liabilities.

**How it was allocated (page 149):** NVIDIA recorded **$14.4 billion of goodwill** (attributable primarily to the workforce and future development of the licensed technology, in the Compute & Networking reporting unit, and tax deductible) and a **$2.5 billion developed technology intangible asset**, valued using a cost-to-recreate methodology with a **five-year useful life**.

The report also notes the arrangement required "significant, nonrefundable payments" and that incorporating the licensed technology into NVIDIA's architectures and product roadmaps carries execution risk (page 102).

*d1:* Decided by d1.1 (TypeSafe AI), minimum confidence 0.87.<br>1. actual_output contains no fact that contradicts expected_output. -> clearly holds (P(yes)=0.96, confidence=0.92, weight=2)<br>2. actual_output states every fact in expected_output. -> clearly holds (P(yes)=0.96, confidence=0.91)<br>3. How closely does actual_output match expected_output? -> "Fully matches" (expected level=0.96 of 1.00, confidence=0.87)<br>Score: 0.96 (weighted mean of 3 applicable questions).

---

**2. How much room does NVIDIA still have to buy back its own shares?**

*Reference:* As of January 25, 2026, NVIDIA was authorized to repurchase up to $58.5 billion of its common stock. On August 26, 2025, the Board approved an additional $60.0 billion of repurchase authorization, without expiration. In fiscal year 2026, NVIDIA repurchased 282 million shares for $40.4 billion.

*Answer:*

**As of January 25, 2026, NVIDIA was authorized to repurchase up to $58.5 billion of its common stock** (pages 121, 131, 164).

Context on how that figure came about:
- On August 26, 2025, the Board of Directors approved an **additional $60.0 billion** in share repurchase authorization, with no expiration date (page 121, 131, 164).
- In fiscal year 2026, NVIDIA repurchased **282 million shares for $40.4 billion** (page 131, 164).
- From **January 26, 2026 through February 20, 2026**, the company repurchased an additional **8 million shares for $1.5 billion** under a pre-established trading plan — which implies roughly $57.0 billion of authorization would remain after that date, though the report states only the $58.5 billion figure as of January 25, 2026 (page 131, 164).

The program may be suspended at any time at the company's discretion, and repurchases are subject to market conditions, applicable legal requirements, and other factors (page 121, 131).

*d1:* Decided by d1.1 (TypeSafe AI), minimum confidence 0.86.<br>1. actual_output contains no fact that contradicts expected_output. -> clearly holds (P(yes)=0.99, confidence=0.98, weight=2)<br>2. actual_output states every fact in expected_output. -> clearly holds (P(yes)=0.97, confidence=0.95)<br>3. How closely does actual_output match expected_output? -> "Fully matches" (expected level=0.95 of 1.00, confidence=0.86)<br>Score: 0.98 (weighted mean of 3 applicable questions).

---

**3. Why did NVIDIA's gross margin go down in fiscal 2026?**

*Reference:* Gross margin fell from 75.0% in fiscal 2025 to 71.1% in fiscal 2026, down 3.9 points. It decreased as the business moved from offering Hopper HGX systems to Blackwell full-scale datacenter solutions. It was also hit by a $4.5 billion charge associated with H20 excess inventory and purchase obligations.

*Answer:*

NVIDIA's gross margin fell to **71.1% in fiscal 2026 from 75.0% in fiscal 2025**, a decline of 3.9 percentage points (page 125, page 128). The report cites two drivers:

1. **Business model transition** — the shift from offering Hopper HGX systems to Blackwell full-scale datacenter solutions (page 129, page 125).
2. **A $4.5 billion charge** in the first quarter of fiscal 2026 associated with H20 excess inventory and purchase obligations (page 129, page 125).

Additional context on inventory effects: total provisions for inventory and excess inventory purchase obligations were $7.2 billion in fiscal 2026 versus $3.7 billion in fiscal 2025, partly offset by releases of $1.5 billion and $689 million, respectively. The net effect of inventory provisions and sales of previously written-down items was an unfavorable impact on gross margin of 2.6% in fiscal 2026 versus 2.3% in fiscal 2025 (page 129, page 126).

*d1:* Decided by d1.1 (TypeSafe AI), minimum confidence 0.93.<br>1. actual_output contains no fact that contradicts expected_output. -> clearly holds (P(yes)=0.99, confidence=0.97, weight=2)<br>2. actual_output states every fact in expected_output. -> clearly holds (P(yes)=0.98, confidence=0.97)<br>3. How closely does actual_output match expected_output? -> "Fully matches" (expected level=0.98 of 1.00, confidence=0.93)<br>Score: 0.98 (weighted mean of 3 applicable questions).

---

**4. How big is NVIDIA's workforce, and how many people work in research?**

*Reference:* At the end of fiscal year 2026, NVIDIA had approximately 42,000 employees in 38 countries. About 31,000 worked in research and development, and 11,000 in sales, marketing, operations and administrative positions. More than 80 percent of the workforce have technical roles, and the turnover rate in fiscal 2026 was 3.7 percent.

*Answer:*

As of the end of fiscal year 2026, NVIDIA had approximately **42,000 employees** across 38 countries. Of those, about **31,000 were engaged in research and development**, while roughly 11,000 worked in sales, marketing, operations, and administrative positions (page 99).

The report also notes that more than 80 percent of the workforce hold technical roles and more than half hold an advanced degree, with a fiscal 2026 turnover rate of 3.7 percent (page 99).

*d1:* Decided by d1.1 (TypeSafe AI), minimum confidence 0.72.<br>1. actual_output contains no fact that contradicts expected_output. -> clearly holds (P(yes)=0.96, confidence=0.93, weight=2)<br>2. actual_output states every fact in expected_output. -> clearly holds (P(yes)=0.86, confidence=0.72)<br>3. How closely does actual_output match expected_output? -> "Fully matches" (expected level=0.93 of 1.00, confidence=0.78)<br>Score: 0.93 (weighted mean of 3 applicable questions).

---

**5. When and where is the 2026 annual shareholder meeting, and who can vote at it?**

*Reference:* The 2026 Annual Meeting of Stockholders is on Wednesday, June 24, 2026, at 9:00 a.m. Pacific Time. It is held virtually at www.virtualshareholdermeeting.com/NVDA2026. Stockholders of record at the close of business on April 27, 2026, can attend and vote.

*Answer:*

NVIDIA’s 2026 Annual Meeting of Stockholders will be held **Wednesday, June 24, 2026, at 9:00 a.m. Pacific Time**, virtually at **www.virtualshareholdermeeting.com/NVDA2026** (pages 20, 25).

Stockholders of record at the close of business on **April 27, 2026** are entitled to vote (pages 20, 25).

*d1:* Decided by d1.1 (TypeSafe AI), minimum confidence 0.84.<br>1. actual_output contains no fact that contradicts expected_output. -> clearly holds (P(yes)=0.98, confidence=0.95, weight=2)<br>2. actual_output states every fact in expected_output. -> clearly holds (P(yes)=0.93, confidence=0.85)<br>3. How closely does actual_output match expected_output? -> "Fully matches" (expected level=0.95 of 1.00, confidence=0.84)<br>Score: 0.96 (weighted mean of 3 applicable questions).

---

...

**11. Which organizations publish the most open-source AI repositories, according to the report?**

*Reference:* A chart titled "NVIDIA is World's Largest Contributor to Open-Source AI" counts repositories on Hugging Face from January 2025 to February 2026. NVIDIA has the most, more than 400. Alibaba Cloud is second, followed by Hugging Face and Ai2 at about 200 each. Meta, Google, IBM, ByteDance, Baidu, Tencent, Microsoft and Z.ai have fewer.

*Answer:*

According to the chart "NVIDIA is World's Largest Contributor to Open-Source AI" on page 7 (source: Hugging Face), repository counts by company grew from zero in January 2025 to these levels by February 2026:

| Company | Repos (Feb 2026) |
|---|---|
| NVIDIA | 420 |
| Alibaba Cloud | 340 |
| Hugging Face | 210 |
| AI2 | ~200 (shown in chart only, not in the table) |
| Meta | 150 |
| Google | 130 |
| IBM | 120 |
| ByteDance | 110 |
| Baidu | 100 |
| Tencent | 85 |
| Microsoft | 80 |
| Z.ai | 45 |

NVIDIA is the largest publisher of open-source AI repositories, followed by Alibaba Cloud, with Hugging Face, AI2, Meta, Google and IBM forming the next tier (page 7).

*d1:* Decided by d1.1 (TypeSafe AI), minimum confidence 0.49.<br>1. actual_output contains no fact that contradicts expected_output. -> likely holds (P(yes)=0.85, confidence=0.69, weight=2)<br>2. actual_output states every fact in expected_output. -> likely holds (P(yes)=0.83, confidence=0.66)<br>3. How closely does actual_output match expected_output? -> "Fully matches", leaning "Mostly right" (expected level=0.83 of 1.00, confidence=0.49)<br>Score: 0.84 (weighted mean of 3 applicable questions).

---

**12. How much cash has NVIDIA returned to shareholders over the past one, three and five years?**

*Reference:* NVIDIA returned $41.4 billion to shareholders over 1 year: $40.4 billion in stock repurchases and $1.0 billion in dividends. Over 3 years it returned $86.3 billion: $84.1 billion in repurchases and $2.2 billion in dividends. Over 5 years it returned $97.2 billion: $94.2 billion in repurchases and $3.0 billion in dividends.

*Answer:*

NVIDIA returned the following amounts of capital to shareholders (page 24):

| Period | Dividends | Stock Repurchases | Total Returned |
|---|---|---|---|
| 1-Year | $1.0B | $40.4B | **$41.4 billion** |
| 3-Year | $2.2B | $84.1B | **$86.3 billion** |
| 5-Year | $3.0B | $94.2B | **$97.2 billion** |

So the vast majority came from stock repurchases rather than dividends. For context, in fiscal 2026 alone NVIDIA repurchased 282 million shares for $40.4 billion and paid $974 million in cash dividends (page 164).

*d1:* Decided by d1.1 (TypeSafe AI), minimum confidence 0.81.<br>1. actual_output contains no fact that contradicts expected_output. -> clearly holds (P(yes)=0.97, confidence=0.95, weight=2)<br>2. actual_output states every fact in expected_output. -> clearly holds (P(yes)=0.94, confidence=0.88)<br>3. How closely does actual_output match expected_output? -> "Fully matches" (expected level=0.94 of 1.00, confidence=0.81)<br>Score: 0.96 (weighted mean of 3 applicable questions).

---

**13. How is the CEO's target pay for fiscal 2026 split between salary, cash and stock awards?**

*Reference:* The CEO's fiscal 2026 target pay mix was 44% SY PSUs, 44% MY PSUs, 8% variable cash and 4% base salary. 96% of the CEO's fiscal 2026 total target pay depended on corporate performance. For the other NEOs, the average target pay mix was 23% SY PSUs, 23% MY PSUs, 46% RSUs, 2% variable cash and 6% base salary.

*Answer:*

For fiscal 2026, CEO Jen-Hsun Huang's target pay totals **$35.5 million**, split as follows (page 65):

| Component | Target ($) | Share of target pay |
|---|---|---|
| Base Salary | 1,500,000 | 4% |
| Variable Cash | 3,000,000 | 8% |
| **Total Cash** | **4,500,000** | **~12%** |
| SY PSUs | 15,500,000 | 44% |
| MY PSUs | 15,500,000 | 44% |
| **Total Equity** | **31,000,000** | **~88%** |
| **TOTAL** | **35,500,000** | **100%** |

So roughly $4.5 million (12.7%) is cash — $1.5 million salary plus a $3.0 million variable cash target — and $31.0 million (87.3%) is stock awards, split evenly between SY PSUs and MY PSUs (pages 57, 65).

Notes from the report:
- Both salary and variable cash were flat with fiscal 2025, while equity rose $3.5 million, or 13%, from the fiscal 2025 target (page 65).
- Targets assume Base Compensation Plan level achievement; actual payouts differ. For fiscal 2026, revenue above the Stretch goal produced a 200% variable cash payout ($6,000,000), and Non-GAAP operating income above the Stretch goal made 150% of the SY PSU target (179,411 shares) eligible to vest (page 65).
- Over 90% of the CEO's total target pay is performance-based and at-risk, and 100% of his equity awards are PSUs (page 27).

*d1:* Decided by d1.1 (TypeSafe AI), minimum confidence 0.31.<br>1. actual_output contains no fact that contradicts expected_output. -> likely holds (P(yes)=0.65, confidence=0.31, weight=2)<br>2. actual_output states every fact in expected_output. -> clearly fails (P(yes)=0.11, confidence=0.79)<br>3. How closely does actual_output match expected_output? -> "Mostly right" (expected level=0.66 of 1.00, confidence=0.65)<br>Score: 0.52 (weighted mean of 3 applicable questions).

---

**14. Who chairs each of the Board's committees, and how often did each committee meet in fiscal 2026?**

*Reference:* The Board has three committees. A. Brooke Seawell chairs the Audit Committee, which met four times in fiscal 2026. Dawn Hudson chairs the Compensation Committee, which met three times. Stephen C. Neal chairs the Nominating and Corporate Governance Committee, which met three times.

*Answer:*

The committee chairs are:

- **Audit Committee:** A. Brooke Seawell, Chairperson (pages 26, 43)
- **Compensation Committee:** Dawn Hudson, Chairperson (page 45)
- **Nominating and Corporate Governance Committee:** Stephen C. Neal, Chairperson (page 45)

**Meetings in fiscal 2026:**
- **Compensation Committee:** met **three times** (page 45)
- **Nominating and Corporate Governance Committee:** met **three times** (page 45)

The retrieved report text does **not state the Audit Committee’s number of meetings**, so I don’t have enough information to give that figure.

*d1:* Decided by d1.1 (TypeSafe AI), minimum confidence 0.61.<br>1. actual_output contains no fact that contradicts expected_output. -> likely holds (P(yes)=0.81, confidence=0.61, weight=2)<br>2. actual_output states every fact in expected_output. -> clearly fails (P(yes)=0.04, confidence=0.92)<br>3. How closely does actual_output match expected_output? -> "Mostly right" (expected level=0.65 of 1.00, confidence=0.80)<br>Score: 0.58 (weighted mean of 3 applicable questions).

---

**15. How much did NVIDIA hold in cash equivalents and marketable securities at the end of fiscal 2026, compared with a year earlier?**

*Reference:* As of January 25, 2026, cash equivalents and marketable securities had an estimated fair value of $65,076 million: $8,285 million reported as cash equivalents, $51,951 million as marketable securities and $4,840 million in other assets. As of January 26, 2025, the total was $42,487 million: $7,866 million in cash equivalents and $34,621 million in marketable securities.

*Answer:*

**NVIDIA's cash, cash equivalents, and marketable securities (pages 130–131):**

| (In millions) | Jan 25, 2026 | Jan 26, 2025 | Change |
|---|---|---|---|
| Cash and cash equivalents | $10,605 | $8,589 | +$2,016 |
| Marketable securities | 51,951 | 34,621 | +$17,330 |
| **Total** | **$62,556** | **$43,210** | **+$19,346** |

So NVIDIA held **$62.6 billion** at the end of fiscal 2026, up from **$43.2 billion** a year earlier — an increase of about **$19.3 billion, or roughly 45%** (page 130). The MD&A liquidity discussion confirms "As of January 25, 2026, we had $62.6 billion in cash, cash equivalents, and marketable securities," with about $1.7 billion held outside the U.S. for which no related foreign or state taxes have been accrued (page 131).

One reconciliation note: Note 7 shows total cash equivalents and marketable securities with an estimated fair value of **$65.1 billion**, because it also includes **$4.8 billion** of long-term publicly-held equity securities (subject to lock-up restrictions through December 2027) that are reported within "Other Assets" rather than in cash/marketable securities on the balance sheet (page 153).

*d1:* Decided by d1.1 (TypeSafe AI), minimum confidence 0.05.<br>1. actual_output contains no fact that contradicts expected_output. -> unclear (P(yes)=0.47, confidence=0.05, weight=2)<br>2. actual_output states every fact in expected_output. -> likely fails (P(yes)=0.24, confidence=0.53)<br>3. How closely does actual_output match expected_output? -> "Mostly right" (expected level=0.59 of 1.00, confidence=0.41)<br>Score: 0.44 (weighted mean of 3 applicable questions).

---